# Semi-Supervised Learning (Air Quality)
Ejecuta el flujo completo de aprendizaje semisupervisado:

1. Carga datos preprocesados.
2. Prepara conjuntos de entrenamiento y prueba, separando datos etiquetados y no etiquetados.
3. Entrena baselines supervisados para tener una referencia sólida.
4. Aplica modelos SSL: Self-Training (con múltiples semillas) y Label Spreading.
5. Evalúa desempeño con métricas estándar, compara resultados y guarda tabla + gráfico final.

Librerias

In [ ]:
import sys
import pickle
import importlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Resolve project root robustly no matter where the notebook is launched from.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

SRC_DIR = ROOT / "src"
DATA_DIR = ROOT / "data"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import evaluation
import models
importlib.reload(evaluation)
importlib.reload(models)

from evaluation import compute_metrics, metrics_to_dataframe
from models import (
    build_label_spreading_model,
    build_self_training_model,
    build_supervised_baseline,
)

Carga de datos preprocesados

In [ ]:
required_pickles = [
    "X_labeled.pkl",
    "y_labeled.pkl",
    "X_unlabeled.pkl",
    "X_test.pkl",
    "y_test.pkl",
]

missing = [name for name in required_pickles if not (DATA_DIR / name).exists()]

if missing:
    processed_path = DATA_DIR / "air_quality_processed.csv"
    if not processed_path.exists():
        raise FileNotFoundError(
            f"No se encontro {processed_path}. Ejecuta primero src/preprocessing.py"
        )

    df = pd.read_csv(processed_path)
    if "target" not in df.columns:
        raise ValueError("El dataset procesado no contiene la columna 'target'")

    drop_cols = [c for c in ["Name", "Start_Date"] if c in df.columns]
    X = df.drop(columns=drop_cols + ["target"])
    X = pd.get_dummies(X, drop_first=False)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(float)
    y = df["target"].astype(int)

    numeric_cols = ["Geo Join ID", "Data Value", "year", "month", "day_of_week"]
    cols_to_scale = [c for c in numeric_cols if c in X.columns]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train.loc[:, cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_test.loc[:, cols_to_scale] = scaler.transform(X_test[cols_to_scale])

    X_labeled, X_unlabeled, y_labeled, _ = train_test_split(
        X_train, y_train, test_size=0.8, random_state=42, stratify=y_train
    )

    with open(DATA_DIR / "X_labeled.pkl", "wb") as f:
        pickle.dump(X_labeled, f)
    with open(DATA_DIR / "y_labeled.pkl", "wb") as f:
        pickle.dump(y_labeled, f)
    with open(DATA_DIR / "X_unlabeled.pkl", "wb") as f:
        pickle.dump(X_unlabeled, f)
    with open(DATA_DIR / "X_test.pkl", "wb") as f:
        pickle.dump(X_test, f)
    with open(DATA_DIR / "y_test.pkl", "wb") as f:
        pickle.dump(y_test, f)

with open(DATA_DIR / "X_labeled.pkl", "rb") as f:
    X_labeled = pickle.load(f)
with open(DATA_DIR / "y_labeled.pkl", "rb") as f:
    y_labeled = pickle.load(f)
with open(DATA_DIR / "X_unlabeled.pkl", "rb") as f:
    X_unlabeled = pickle.load(f)
with open(DATA_DIR / "X_test.pkl", "rb") as f:
    X_test = pickle.load(f)
with open(DATA_DIR / "y_test.pkl", "rb") as f:
    y_test = pickle.load(f)

Baseline para comparación

In [ ]:
rows = []

baseline = build_supervised_baseline(random_state=42)
baseline.fit(X_labeled, y_labeled)
y_pred_base = baseline.predict(X_test)
rows.append({"model": "SupervisedBaseline(RandomForest)", **compute_metrics(y_test, y_pred_base)})

Self-Training (Pseudo-labeling)

In [ ]:
def self_training(
    model,
    X_labeled,
    y_labeled,
    X_unlabeled,
    X_test,
    y_test,
    threshold=0.85,
    max_iterations=5,
    verbose=True,
):
    """
    Self-training iterativo con pseudo-etiquetado por confianza.
    Retorna el modelo final y el historial por iteración.
    """
    X_train = X_labeled.copy()
    y_train = y_labeled.copy()
    X_pool = X_unlabeled.copy()

    history = []

    for iteration in range(1, max_iterations + 1):
        if len(X_pool) == 0:
            if verbose:
                print(f"Iteracion {iteration}: no quedan datos sin etiqueta")
            break

        model.fit(X_train, y_train)
        y_proba_pool = model.predict_proba(X_pool)
        confidence = np.max(y_proba_pool, axis=1)
        high_conf_mask = confidence >= threshold
        new_labels = int(np.sum(high_conf_mask))

        if verbose:
            print(f"Iteracion {iteration}: {new_labels}/{len(X_pool)} pseudo-etiquetas nuevas")

        if new_labels == 0:
            break

        X_new = X_pool.loc[high_conf_mask]
        y_new = pd.Series(model.predict(X_new), index=X_new.index)

        X_train = pd.concat([X_train, X_new], axis=0)
        y_train = pd.concat([y_train, y_new], axis=0)
        X_pool = X_pool.loc[~high_conf_mask]

        y_pred_test = model.predict(X_test)
        history.append(
            {
                "iteration": iteration,
                "new_labels": new_labels,
                "pool_remaining": int(len(X_pool)),
                "accuracy": accuracy_score(y_test, y_pred_test),
                "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),
            }
        )

    model.fit(X_train, y_train)
    return model, history

# Pool SSL para métodos basados en -1 (scikit-learn API)
X_ssl = pd.concat([X_labeled, X_unlabeled], axis=0)
y_ssl = np.concatenate([y_labeled.to_numpy(), np.full(len(X_unlabeled), -1, dtype=int)])

# Baseline adicional con Logistic Regression (sobre datos etiquetados)
lr_baseline = LogisticRegression(max_iter=1500, class_weight="balanced", random_state=42)
lr_baseline.fit(X_labeled, y_labeled)
y_pred_lr = lr_baseline.predict(X_test)
rows.append({"model": "SupervisedBaseline(LogisticRegression)", **compute_metrics(y_test, y_pred_lr)})

# Self-training con multiples semillas
seeds = [42, 123, 456]
results_per_seed = {}
seed_metric_rows = []

for seed in seeds:
    ssl_model = LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        random_state=seed,
    )

    model_ssl, history = self_training(
        ssl_model,
        X_labeled=X_labeled,
        y_labeled=y_labeled,
        X_unlabeled=X_unlabeled,
        X_test=X_test,
        y_test=y_test,
        threshold=0.85,
        max_iterations=5,
        verbose=False,
    )

    y_pred_ssl = model_ssl.predict(X_test)
    metrics = compute_metrics(y_test, y_pred_ssl)
    results_per_seed[seed] = {"metrics": metrics, "history": history}
    seed_metric_rows.append(metrics)

# Guardamos una fila agregada para Self-Training
rows.append(
    {
        "model": "SemiSupervised(SelfTraining)_mean_3seeds",
        "accuracy": float(np.mean([m["accuracy"] for m in seed_metric_rows])),
        "precision_weighted": float(np.mean([m["precision_weighted"] for m in seed_metric_rows])),
        "recall_weighted": float(np.mean([m["recall_weighted"] for m in seed_metric_rows])),
        "f1_weighted": float(np.mean([m["f1_weighted"] for m in seed_metric_rows])),
        "std_accuracy": float(np.std([m["accuracy"] for m in seed_metric_rows])),
        "std_f1_weighted": float(np.std([m["f1_weighted"] for m in seed_metric_rows])),
    }
)

Label Spreading

In [ ]:
label_spread = build_label_spreading_model()
label_spread.fit(X_ssl, y_ssl)
y_pred_label_spread = label_spread.predict(X_test)
rows.append({"model": "SemiSupervised(LabelSpreading)", **compute_metrics(y_test, y_pred_label_spread)})

Mostrar y Guardar Resultados (tablas, logs, reproducibilidad)

In [ ]:
results_df = metrics_to_dataframe(rows)
results_path = DATA_DIR / "ssl_results.csv"
results_df.to_csv(results_path, index=False)

# Comparacion visual baseline LR vs Self-Training promedio
baseline_lr_row = results_df[results_df["model"] == "SupervisedBaseline(LogisticRegression)"]
ssl_row = results_df[results_df["model"] == "SemiSupervised(SelfTraining)_mean_3seeds"]

if not baseline_lr_row.empty and not ssl_row.empty:
    baseline_lr_acc = float(baseline_lr_row["accuracy"].iloc[0])
    ssl_acc_mean = float(ssl_row["accuracy"].iloc[0])

    plt.figure(figsize=(7, 4))
    sns.barplot(
        x=["Baseline LR", "Self-Training (mean 3 seeds)"],
        y=[baseline_lr_acc, ssl_acc_mean],
        palette=["#d95f02", "#1b9e77"],
    )
    plt.ylim(0, 1)
    plt.ylabel("Accuracy")
    plt.title("Comparacion Baseline vs SSL")
    plt.tight_layout()
    plt.savefig(DATA_DIR / "ssl_comparison.png", dpi=200)
    plt.show()

print(f"Resultados guardados en: {results_path}")
print(f"Grafico guardado en: {DATA_DIR / 'ssl_comparison.png'}")
results_df